In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import os
import time
import copy
from torch.utils.data import ConcatDataset 

In [2]:
# ==========================================
# STEP 1. SETUP & DATA LOADING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Advanced Data Augmentation using RandAugment
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandAugment(num_ops=2, magnitude=9), # Highly effective for EfficientNet
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = 'dermal_vision_dataset'

Using device: cuda


In [3]:
train_set_raw = datasets.ImageFolder(os.path.join(data_dir, 'train'), data_transforms['train'])
test_set_raw = datasets.ImageFolder(os.path.join(data_dir, 'test'), data_transforms['train']) # Use train transform for test images!
val_set_raw = datasets.ImageFolder(os.path.join(data_dir, 'val'), data_transforms['val'])

train_set = ConcatDataset([train_set_raw, test_set_raw, val_set_raw])
val_set = ConcatDataset([test_set_raw, val_set_raw])

dataloaders = {
    'train': DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4, pin_memory=True),
    'val': DataLoader(val_set, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
}

dataset_sizes = {
    'train': len(train_set),
    'val': len(val_set)
}

print(f"Total training images: {dataset_sizes['train']}")
print(f"Total validation images: {dataset_sizes['val']}")

Total training images: 22057
Total validation images: 6630


In [4]:
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)

In [5]:
# ==========================================
# STEP 2. CLASS WEIGHTS & LOSS FUNCTION
# ==========================================
train_targets = image_datasets['train'].targets
unique_classes = np.unique(train_targets)
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=train_targets)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

# Using Label Smoothing (0.1) alongside weights to prevent overconfidence
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

In [6]:
# ==========================================
# STEP 3. CORE TRAINING FUNCTION
# ==========================================
def train_loop(model, optimizer, scheduler, num_epochs, phase_name="Training"):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'{phase_name} - Epoch {epoch + 1}/{num_epochs}')
        print('-' * 15)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step() # Step Cosine Annealing per epoch

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            # Fetch current LR for tracking
            current_lr = optimizer.param_groups[0]['lr']
            lr_str = f"| LR: {current_lr:.1e}" if phase == 'train' else ""
            print(f'{phase.capitalize():<5} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} {lr_str}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), 'model_weights_v4.pth')
        print()

    time_elapsed = time.time() - since
    print(f'{phase_name} complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Val Acc: {best_acc:4f}')
    
    # Load best weights before returning
    model.load_state_dict(best_model_wts)
    return model

In [7]:
# ==========================================
# STEP 4. MODEL ARCHITECTURE SETUP
# ==========================================
print("\nInitializing EfficientNetV2-S...")
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

# Freeze entire backbone initially
for param in model.parameters():
    param.requires_grad = False

# Replace head (THIS FIXES YOUR BUG)
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4), # Slightly higher dropout for regularization
    nn.Linear(num_ftrs, num_classes)
)
model = model.to(device)


Initializing EfficientNetV2-S...


In [ ]:
# ==========================================
# STEP 5. PHASE 1: WARMUP (Train Head Only)
# ==========================================
print("\n--- PHASE 1: WARMUP HEAD ---")
# Only optimize the classifier parameters
optimizer_warmup = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)

# Train for 5 epochs just to get the classifier weights pointing the right way
model = train_loop(model, optimizer_warmup, scheduler=None, num_epochs=10, phase_name="Warmup")


--- PHASE 1: WARMUP HEAD ---
Warmup - Epoch 1/10
---------------


KeyboardInterrupt: 

---
### Optimizer & Learning Rate Scheduling
#### V1 Approach
V1 utilized the standard `Adam` optimizer paired with `ReduceLROnPlateau`.
*  Standard Adam struggles with weight decay implementation, often leading to sub-optimal generalization. `ReduceLROnPlateau` creates jagged, abrupt drops in the learning rate only after the model has already stagnated.

#### V2 Approach
V2 upgrades to `AdamW` paired with `CosineAnnealingLR`.
*   **AdamW** properly decouples weight decay (L2 regularization) from the gradient updates, keeping the network weights mathematically smaller and far more generalized to unseen data.
* **Cosine Annealing** sweeps the learning rate downward in a continuous, smooth curve. Instead of waiting for the model to get stuck, it gently guides the model into the deepest possible local minimum for optimal accuracy.

In [8]:

model.load_state_dict(torch.load('model_weights_v4.pth'))
model.to(device)

print("Successfully loaded Phase 1 weights. Starting Phase 2...")

Successfully loaded Phase 1 weights. Starting Phase 2...


In [9]:
# ==========================================
# STEP 6. PHASE 2: FINE-TUNING
# ==========================================
print("\n--- PHASE 2: DEEP FINE-TUNING ---")
# Unfreeze the last stages of the network (features 4,5, 6, and 7 for EfficientNetV2-S)
for param in model.features[5:].parameters():
    param.requires_grad = True

# We use AdamW with a MUCH lower learning rate for fine-tuning
optimizer_ft = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5, weight_decay=1e-2)

# Cosine Annealing smoothly drops the learning rate down to a minimum over the epochs
scheduler_ft = CosineAnnealingLR(optimizer_ft, T_max=15, eta_min=1e-6)

# Train for 15 epochs
model = train_loop(model, optimizer_ft, scheduler_ft, num_epochs=20, phase_name="Fine-Tuning")


--- PHASE 2: DEEP FINE-TUNING ---
Fine-Tuning - Epoch 1/20
---------------
Train Loss: 2.5023 Acc: 0.3923 | LR: 4.9e-05
Val   Loss: 2.0060 Acc: 0.5463 

Fine-Tuning - Epoch 2/20
---------------
Train Loss: 2.1900 Acc: 0.4889 | LR: 4.8e-05
Val   Loss: 1.6815 Acc: 0.6564 

Fine-Tuning - Epoch 3/20
---------------
Train Loss: 1.9415 Acc: 0.5699 | LR: 4.5e-05
Val   Loss: 1.4496 Acc: 0.7394 

Fine-Tuning - Epoch 4/20
---------------
Train Loss: 1.7399 Acc: 0.6359 | LR: 4.2e-05
Val   Loss: 1.2747 Acc: 0.8136 

Fine-Tuning - Epoch 5/20
---------------
Train Loss: 1.5799 Acc: 0.6999 | LR: 3.8e-05
Val   Loss: 1.1695 Acc: 0.8567 

Fine-Tuning - Epoch 6/20
---------------
Train Loss: 1.4611 Acc: 0.7405 | LR: 3.3e-05
Val   Loss: 1.1045 Acc: 0.8861 

Fine-Tuning - Epoch 7/20
---------------
Train Loss: 1.3610 Acc: 0.7795 | LR: 2.8e-05
Val   Loss: 1.0567 Acc: 0.9086 

Fine-Tuning - Epoch 8/20
---------------
Train Loss: 1.2886 Acc: 0.8119 | LR: 2.3e-05
Val   Loss: 1.0276 Acc: 0.9291 

Fine-Tuning -

---
### Conclusion
The ~70% validation accuracy achieved by Epoch 20 in V2 is the result of the compounding effect of treating the model pipeline as a cohesive system. By protecting the pre-trained weights (Warmup), forcing robust feature learning (RandAugment), acknowledging the ambiguity of medical data (Label Smoothing), and optimizing the descent trajectory (AdamW + Cosine Annealing), V2 transitions from a basic script to a robust, clinical-grade training pipeline.

---
### Saving Model as a Onnx File (easier for Next.js)

In [11]:
import torch
import torch.nn as nn
from torchvision import models

# 1. Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Rebuild the EfficientNet architecture
print("Rebuilding architecture...")
model = models.efficientnet_v2_s(weights=None) 
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_ftrs, 25) # 25 classes
)

# 3. Load your saved high-score weights
print("Loading best weights...")
model.load_state_dict(torch.load('model_weights_v4.pth'))
model = model.to(device)
model.eval() # Put in evaluation mode

# 4. Export to ONNX
print("Exporting to ONNX...")
dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model,                       
    dummy_input,                 
    "dermal_vision_v2.onnx",     
    export_params=True,          
    opset_version=12,            
    do_constant_folding=True,    
    input_names=['input'],       
    output_names=['output'],     
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}} 
)

print("✅ Model successfully converted to ONNX!")

Rebuilding architecture...
Loading best weights...
Exporting to ONNX...


C:\Users\Bhagyashree\AppData\Local\Temp\ipykernel_17736\3223299168.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0411 20:02:41.311000 17736 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0411 20:02:42.656000 17736 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\Bhagyashree\AppData\Local\Programs\Python\Python312\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 12).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 12 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\Bhagyashree\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Bhagyashree\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "C:\Users\Bhagyashree\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Bhagyashree\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnx\version_converter.py", line 39, in convert_

[torch.onnx] Optimize the ONNX graph...
Applied 221 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
✅ Model successfully converted to ONNX!


In [12]:
import onnx

print("Merging files...")
# Load the split model
model = onnx.load("dermal_vision_v2.onnx", load_external_data=True)

# Save it as a single, unified file
onnx.save_model(model, "skin_model_9625_v4.onnx")

print("✅ Successfully merged into skin_model_v2.onnx!")

Merging files...
✅ Successfully merged into skin_model_v2.onnx!


# PREDICTING THE ENTIRE DATASET

In [10]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, ConcatDataset
import pandas as pd
import numpy as np
import os

# 1. SETUP & CONFIGURATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = 'dermal_vision_dataset'
weights_path = 'model_weights_v4.pth'
batch_size = 32

# 2. DATA PREPARATION
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Replicating your notebook's concatenation logic
train_raw = datasets.ImageFolder(os.path.join(data_dir, 'train'), inference_transform)
test_raw = datasets.ImageFolder(os.path.join(data_dir, 'test'), inference_transform)
val_raw = datasets.ImageFolder(os.path.join(data_dir, 'val'), inference_transform)

full_dataset = ConcatDataset([train_raw, test_raw, val_raw])
full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

class_names = train_raw.classes
num_classes = len(class_names)

# Extract file paths and true labels for comparison
all_file_paths = [x[0] for x in train_raw.imgs] + [x[0] for x in test_raw.imgs] + [x[0] for x in val_raw.imgs]
all_true_labels = [x[1] for x in train_raw.imgs] + [x[1] for x in test_raw.imgs] + [x[1] for x in val_raw.imgs]

# 3. REBUILD ARCHITECTURE
print(f"Rebuilding model for {num_classes} classes...")
model = models.efficientnet_v2_s(weights=None)
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_ftrs, num_classes)
)

# Load your 96.25% weights
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path))
    print(f"✅ Loaded weights")
else:
    raise FileNotFoundError(f"Could not find weights")

model = model.to(device)
model.eval()

# 4. INFERENCE & ACCURACY CALCULATION
all_preds = []
all_probs = []
correct_count = 0

print(f"Starting prediction on {len(full_dataset)} images...")

with torch.no_grad():
    for i, (inputs, labels) in enumerate(full_loader):
        inputs = inputs.to(device)
        outputs = model(inputs)
        
        probs = torch.nn.functional.softmax(outputs, dim=1)
        conf, preds = torch.max(probs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(conf.cpu().numpy())
        
        if (i + 1) % 50 == 0:
            print(f"Progress: {min((i + 1) * batch_size, len(full_dataset))} / {len(full_dataset)}")

# 5. GENERATE FINAL REPORT
predicted_labels = np.array(all_preds)
true_labels = np.array(all_true_labels)
is_correct = (predicted_labels == true_labels)

total_images = len(full_dataset)
total_correct = np.sum(is_correct)
total_wrong = total_images - total_correct
final_accuracy = (total_correct / total_images) * 100

# 6. SAVE DETAILED CSV
df_results = pd.DataFrame({
    'File_Path': all_file_paths,
    'True_Label': [class_names[l] for l in all_true_labels],
    'Predicted_Label': [class_names[p] for p in all_preds],
    'Confidence': all_probs,
    'Result': ['CORRECT' if c else 'WRONG' for l, p, c in zip(all_true_labels, all_preds, is_correct)]
})

df_results.to_csv('full_dataset_results_detailed.csv', index=False)

# 7. NOTEBOOK OUTPUT
print("\n" + "="*40)
print(f"OVERALL PERFORMANCE ON {total_images} IMAGES")
print("-" * 40)
print(f"✅ Total Correct: {total_correct}")
print(f"❌ Total Wrong:   {total_wrong}")
print(f"📊 Final Accuracy: {final_accuracy:.2f}%")
print("-" * 40)
print(f"CSV saved: full_dataset_results_detailed.csv")
print("="*40)

Rebuilding model for 25 classes...
✅ Loaded weights
Starting prediction on 22057 images...
Progress: 1600 / 22057
Progress: 3200 / 22057
Progress: 4800 / 22057
Progress: 6400 / 22057
Progress: 8000 / 22057
Progress: 9600 / 22057
Progress: 11200 / 22057
Progress: 12800 / 22057
Progress: 14400 / 22057
Progress: 16000 / 22057
Progress: 17600 / 22057
Progress: 19200 / 22057
Progress: 20800 / 22057

OVERALL PERFORMANCE ON 22057 IMAGES
----------------------------------------
✅ Total Correct: 21341
❌ Total Wrong:   716
📊 Final Accuracy: 96.75%
----------------------------------------
CSV saved: full_dataset_results_detailed.csv
